# Citations, decoded — the `<r>`/`<c>` span pipeline

`walkthrough.ipynb` traced the **guardian** path, whose output is a single yes/no token scored from
logprobs. The **citations** adapter is the opposite end of the complexity range: the model never sees
or emits raw text spans — it emits **sentence index pairs**, and Mellea's `IntrinsicsResultProcessor`
decodes those indices back into character spans in the original answer and documents.

This notebook runs one citations call **layer by layer** and prints the artifact each stage produces:

1. The `io.yaml` — `sentence_boundaries` + a 6-stage `transformations` pipeline.
2. `IntrinsicsRewriter` tags the answer sentences (`<r0>`, `<r1>`…) and document sentences (`<c0>`, `<c1>`…).
3. The model emits `[{"r": i, "c": [j, …]}, …]` — pure indices.
4. The processor pipeline (`explode → drop_duplicates → decode_sentences ×2 → project → merge_spans`)
   turns those indices into `{response_text, citation_doc_id, citation_text, …begin/end}` records.

Unlike guardian (which the bridge scores inline), citations goes through Mellea's **real**
`IntrinsicsResultProcessor`, because the structural transforms need the rewritten request's
sentence-marked documents to resolve indices → spans. That's the `_needs_processor` branch in
`ollama_intrinsic.py`.

## Prerequisites

Same as the other notebooks: patched `ollama serve` with `granite-switch`, the GGUF locally, project
venv. The citations io.yaml is fetched from the HF RAG library on first use and cached in
`.rag_io_cache/` (already cached in this repo).

## 1 · The citations `io.yaml`

Two fields make citations special. **`sentence_boundaries`** tells the rewriter to insert sentence
markers: `r` in the last message (the answer → `<r0>`, `<r1>`…) and `c` in the documents (`<c0>`,
`<c1>`…, numbered continuously across all docs). **`transformations`** is a 6-stage pipeline that
decodes the model's index output back into spans.

In [1]:
import json, os
from ollama_intrinsic import load_io_config

cfg = load_io_config("citations")

print("sentence_boundaries:", cfg["sentence_boundaries"])
print("\ntransformation pipeline (applied in order):")
for i, t in enumerate(cfg["transformations"], 1):
    extra = {k: v for k, v in t.items() if k not in ("type",)}
    print(f"  {i}. {t['type']:16} {extra}")

sentence_boundaries: {'last_message': 'r', 'documents': 'c'}

transformation pipeline (applied in order):
  1. explode          {'input_path': [], 'target_field': 'c'}
  2. drop_duplicates  {'input_path': [], 'target_fields': ['r', 'c']}
  3. decode_sentences {'source': 'last_message', 'input_path': [None, 'r'], 'output_names': {'begin': 'response_begin', 'end': 'response_end', 'text': 'response_text'}}
  4. decode_sentences {'source': 'documents', 'input_path': [None, 'c'], 'output_names': {'document_id': 'citation_doc_id', 'begin': 'citation_begin', 'end': 'citation_end', 'text': 'citation_text'}}
  5. project          {'input_path': [], 'retained_fields': ['response_begin', 'response_end', 'response_text', 'citation_doc_id', 'citation_begin', 'citation_end', 'citation_text']}
  6. merge_spans      {'input_path': [], 'group_fields': ['response_begin', 'response_end', 'response_text', 'citation_doc_id'], 'begin_field': 'citation_begin', 'end_field': 'citation_end', 'text_field': 'cita

## 2 · Inputs: an answer + the documents it draws on

We use a two-sentence answer where the second sentence is a **paraphrase** of a document sentence
("It sits on the Seine river." ← "Paris is located on the Seine river."). That makes the index→span
decoding visible: the citation text won't be a substring of the answer.

In [2]:
documents = [
    {"doc_id": "0", "text": "The capital of France is Paris. Paris is located on the Seine river."},
    {"doc_id": "1", "text": "Mount Everest is the tallest mountain on Earth, at 8,849 meters."},
]
answer = "The capital of France is Paris. It sits on the Seine river."

# Citations classifies the whole turn: user question + the assistant answer to cite.
ctx = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": answer},
]
print("answer to cite:", answer)

answer to cite: The capital of France is Paris. It sits on the Seine river.


## 3 · Rewriter — insert the `<r>` / `<c>` sentence markers

`IntrinsicsRewriter.transform` splits the answer and documents into sentences (via nltk) and prefixes
each with a numbered marker. It also appends the instruction message telling the model to emit index
pairs. Watch where the markers land:

- the **assistant** message gets `<r0> … <r1> …`
- each **document** gets `<c0> … <c1> …`, numbered continuously across docs (so doc 1 starts at `<c2>`).

In [3]:
from mellea.formatters import granite as g

rewriter = g.IntrinsicsRewriter(config_dict=cfg, model_name="citations")
rewritten = rewriter.transform({"messages": list(ctx), "extra_body": {"documents": documents}})

print("=== rewritten messages ===")
for i, m in enumerate(rewritten.messages):
    body = m.model_dump(exclude_unset=True).get("content", "")
    print(f"\n[{i}] {m.role}:")
    print(f"    {body[:200]}{'…' if len(body) > 200 else ''}")

print("\n=== rewritten documents (sentence-tagged) ===")
for d in rewritten.extra_body.documents:
    print(f"  doc {d.doc_id}: {d.text}")

=== rewritten messages ===

[0] user:
    What is the capital of France?

[1] assistant:
    <r0> The capital of France is Paris. <r1> It sits on the Seine river.

[2] user:
    Split the last assistant response into individual sentences.  For each sentence in the response, identify the statement IDs from the below  documents that it references. Ensure that your output includ…

=== rewritten documents (sentence-tagged) ===
  doc 0: <c0> The capital of France is Paris. <c1> Paris is located on the Seine river.
  doc 1: <c2> Mount Everest is the tallest mountain on Earth, at 8,849 meters.


## 4 · Render + generate — the model emits **indices**, not text

We render the GGUF chat template with the `citations` control token over the *rewritten* (tagged)
messages and documents, then POST to Ollama raw. The model's job is purely to match response
sentences to document sentences by number — its output is a compact list of `{r, c}` pairs.

In [4]:
from ollama_intrinsic import OllamaIntrinsicBackend

MODEL = os.environ.get("GRANITE_SWITCH_MODEL", "granite-switch")
GGUF = os.environ.get("GRANITE_SWITCH_GGUF", "granite-switch-4.1-3b-preview-f16.gguf")
backend = OllamaIntrinsicBackend(model=MODEL, gguf_path=GGUF)

rendered_messages = [
    {"role": m.role, "content": m.model_dump(exclude_unset=True).get("content")}
    for m in rewritten.messages
]
rendered_documents = [{"doc_id": d.doc_id, "text": d.text} for d in rewritten.extra_body.documents]

prompt = backend.render(rendered_messages, adapter_name="citations", documents=rendered_documents)
resp = backend.generate(prompt, num_predict=4096)

print("raw model output (sentence-index pairs):")
print("   ", resp["response"].strip())
print("\nread it as: response-sentence r references document-sentence(s) c")

raw model output (sentence-index pairs):
    [{"r": 0, "c": [0]}, {"r": 1, "c": [1]}]

read it as: response-sentence r references document-sentence(s) c


## 5 · Processor — decode indices → character spans

Now Mellea's `IntrinsicsResultProcessor` runs the 6-stage pipeline over that index output, using the
**rewritten** request (whose tagged documents and answer let it resolve each index back to its span):

- **`explode`** — a pair like `{r:0, c:[0,1]}` becomes one row per cited doc sentence.
- **`drop_duplicates`** — the model sometimes repeats pairs; dedupe.
- **`decode_sentences` (last_message)** — replace each `r` index with the answer sentence's `begin/end/text`.
- **`decode_sentences` (documents)** — replace each `c` index with the doc's `doc_id/begin/end/text`.
- **`project`** — keep just the seven span fields.
- **`merge_spans`** — fuse adjacent document spans cited by the same response sentence.

The bridge wraps Ollama's plain text in the `ChatCompletionResponse` shape the processor expects (its
`_run_processor` helper) and hands it the rewritten request.

In [5]:
from ollama_intrinsic import _run_processor

processor = g.IntrinsicsResultProcessor(config_dict=cfg)
decoded = _run_processor(processor, resp["response"].strip(), rewritten)

print(json.dumps(decoded, indent=2, default=str))

[
  {
    "response_begin": 0,
    "response_end": 32,
    "response_text": "The capital of France is Paris. ",
    "citation_doc_id": "0",
    "citation_begin": 0,
    "citation_end": 32,
    "citation_text": "The capital of France is Paris. "
  },
  {
    "response_begin": 32,
    "response_end": 59,
    "response_text": "It sits on the Seine river.",
    "citation_doc_id": "0",
    "citation_begin": 32,
    "citation_end": 68,
    "citation_text": "Paris is located on the Seine river."
  }
]


## 6 · Read the decoded citations

Each record links a span of the **answer** to the supporting span in a **document**. Note the second
citation: the answer says *"It sits on the Seine river."* but the cited document text is *"Paris is
located on the Seine river."* — the model resolved a paraphrase by sentence index, and the processor
recovered the *actual document wording* and its character offsets. That's why citations decodes
indices rather than trusting string matching.

In [6]:
for i, c in enumerate(decoded, 1):
    print(f"citation {i}:")
    print(f"  answer span [{c['response_begin']}:{c['response_end']}]  {c['response_text']!r}")
    print(f"   ← doc {c['citation_doc_id']} [{c['citation_begin']}:{c['citation_end']}]  {c['citation_text']!r}")
    # Verify the offsets actually point where they claim:
    doc_text = next(d['text'] for d in documents if d['doc_id'] == c['citation_doc_id'])
    sliced = doc_text[c['citation_begin']:c['citation_end']]
    print(f"     check: doc_text[{c['citation_begin']}:{c['citation_end']}] == {sliced!r}")
    print()

citation 1:
  answer span [0:32]  'The capital of France is Paris. '
   ← doc 0 [0:32]  'The capital of France is Paris. '
     check: doc_text[0:32] == 'The capital of France is Paris. '

citation 2:
  answer span [32:59]  'It sits on the Seine river.'
   ← doc 0 [32:68]  'Paris is located on the Seine river.'
     check: doc_text[32:68] == 'Paris is located on the Seine river.'



## 7 · The one-line equivalent

Everything in §3–5 is exactly what `backend.call_adapter("citations", …)` does internally. The result
is identical to the step-by-step decode above.

In [7]:
out = backend.call_adapter("citations", ctx, documents=documents, num_predict=4096)
oneline = out.get("parsed") or []
print("matches the step-by-step decode:", oneline == decoded)
print(json.dumps(oneline, indent=2, default=str))

matches the step-by-step decode: True
[
  {
    "response_begin": 0,
    "response_end": 32,
    "response_text": "The capital of France is Paris. ",
    "citation_doc_id": "0",
    "citation_begin": 0,
    "citation_end": 32,
    "citation_text": "The capital of France is Paris. "
  },
  {
    "response_begin": 32,
    "response_end": 59,
    "response_text": "It sits on the Seine river.",
    "citation_doc_id": "0",
    "citation_begin": 32,
    "citation_end": 68,
    "citation_text": "Paris is located on the Seine river."
  }
]


## Recap: guardian vs citations

| | guardian-core (`walkthrough.ipynb`) | citations (this notebook) |
|---|---|---|
| Model emits | one token: `"yes"` / `"no"` | index pairs `[{r, c}, …]` |
| io.yaml signal | `likelihood` transformation → needs **logprobs** | `sentence_boundaries` + structural transforms → needs the **rewritten request** |
| Decoded by | inline logprob scoring (bridge) | Mellea's real `IntrinsicsResultProcessor` (`_needs_processor` branch) |
| Output | a calibrated float score | answer-span → document-span records |

Both are the **same Granite Switch model**, switched by a control token, with **Mellea's own io.yaml +
rewriter/processor** producing the envelope and decoding the output — just running over Ollama instead
of vLLM.